In [2]:
# NOTEBOOK: 03_spark.ipynb  -- cell 1 (set Java, then start Spark)
import os
os.environ["JAVA_HOME"] = "/home/jovyan/.jdk/jdk-17.0.20+8"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("receipt-pipeline")
         .master("local[*]")
         .getOrCreate())

print("Spark version:", spark.version)
print(spark.range(5).toPandas())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/jovyan/receipt-env/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/10 18:13:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


/home/jovyan/receipt-env/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


   id
0   0
1   1
2   2
3   3
4   4


In [3]:
# NOTEBOOK: 03_spark.ipynb  -- cell 2 (load the 800 OCR files)
df = spark.read.json("data/interim/train/*.json")

print("rows (one per receipt):", df.count())
df.printSchema()

26/08/10 18:15:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/interim/train/*.json.
java.io.FileNotFoundException: File data/interim/train/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.anal

rows (one per receipt): 800
root
 |-- ground_truth: string (nullable = true)
 |-- idx: long (nullable = true)
 |-- ocr: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- box: array (nullable = true)
 |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |-- conf: double (nullable = true)
 |    |    |-- text: string (nullable = true)



In [4]:
# NOTEBOOK: 03_spark.ipynb  -- cell 3 (explode to one row per word)
from pyspark.sql import functions as F

words = (df
    .select("idx", F.explode("ocr").alias("w"))   # one row per OCR word
    .select(
        "idx",
        F.col("w.text").alias("text"),
        F.col("w.conf").alias("conf"),
        F.col("w.box").alias("box"),
    ))

print("total word rows:", words.count())
words.show(10, truncate=False)

total word rows: 14715
+---+----------+-------------------+----------------------------------------------------------------+
|idx|text      |conf               |box                                                             |
+---+----------+-------------------+----------------------------------------------------------------+
|0  |Nasi      |0.9998831152915955 |[[300.0, 366.0], [354.0, 366.0], [354.0, 392.0], [300.0, 392.0]]|
|0  |Campur    |0.992746930021226  |[[362.0, 366.0], [440.0, 366.0], [440.0, 390.0], [362.0, 390.0]]|
|0  |Bali      |0.9818131327629089 |[[446.0, 364.0], [498.0, 364.0], [498.0, 388.0], [446.0, 388.0]]|
|0  |75,000    |0.41665860322926634|[[542.0, 360.0], [618.0, 360.0], [618.0, 386.0], [542.0, 386.0]]|
|0  |Bbk Bengil|0.8182316162725418 |[[304.0, 392.0], [428.0, 392.0], [428.0, 420.0], [304.0, 420.0]]|
|0  |Nasi      |0.8508836030960083 |[[436.0, 392.0], [488.0, 392.0], [488.0, 416.0], [436.0, 416.0]]|
|0  |125,000   |0.37147005657504384|[[532.0, 388.0], [618.0

In [5]:
# NOTEBOOK: 03_spark.ipynb  -- cell 4 (clean and normalize)
from pyspark.sql import functions as F

clean = (words
    # 1. trim whitespace, drop empty/near-empty tokens
    .withColumn("text", F.trim("text"))
    .filter(F.length("text") > 0)

    # 2. flag low-confidence reads instead of deleting them
    #    (deleting hides OCR failures; flagging is honest and lets you measure them)
    .withColumn("low_conf", F.col("conf") < 0.5)

    # 3. detect price-like tokens: digits with , or . (e.g. 75,000)
    #    strip separators to a clean integer string where possible
    .withColumn("is_price_like",
        F.col("text").rlike(r"^[0-9][0-9.,]*$"))
    .withColumn("price_num",
        F.when(F.col("is_price_like"),
               F.regexp_replace("text", r"[.,]", "")).otherwise(None))
)

clean.show(10, truncate=False)

print("total rows:", clean.count())
print("low-confidence rows:", clean.filter("low_conf").count())
print("price-like rows:", clean.filter("is_price_like").count())

+---+----------+-------------------+----------------------------------------------------------------+--------+-------------+---------+
|idx|text      |conf               |box                                                             |low_conf|is_price_like|price_num|
+---+----------+-------------------+----------------------------------------------------------------+--------+-------------+---------+
|0  |Nasi      |0.9998831152915955 |[[300.0, 366.0], [354.0, 366.0], [354.0, 392.0], [300.0, 392.0]]|false   |false        |NULL     |
|0  |Campur    |0.992746930021226  |[[362.0, 366.0], [440.0, 366.0], [440.0, 390.0], [362.0, 390.0]]|false   |false        |NULL     |
|0  |Bali      |0.9818131327629089 |[[446.0, 364.0], [498.0, 364.0], [498.0, 388.0], [446.0, 388.0]]|false   |false        |NULL     |
|0  |75,000    |0.41665860322926634|[[542.0, 360.0], [618.0, 360.0], [618.0, 386.0], [542.0, 386.0]]|true    |true         |75000    |
|0  |Bbk Bengil|0.8182316162725418 |[[304.0, 392.0], [4

total rows: 14687


low-confidence rows: 4814


price-like rows: 4260


In [6]:
# NOTEBOOK: 03_spark.ipynb  -- cell 5 (parse the ground-truth total, one row per receipt)
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import json

# ground_truth is a JSON string. Pull out the grand total (total.total_price).
def extract_total(gt_string):
    try:
        gt = json.loads(gt_string)["gt_parse"]
        return gt.get("total", {}).get("total_price")
    except Exception:
        return None

extract_total_udf = F.udf(extract_total, StringType())

labels = (df
    .select("idx", "ground_truth")
    .withColumn("true_total_raw", extract_total_udf("ground_truth"))
    # normalize the label the same way as the OCR prices: strip , and .
    .withColumn("true_total",
        F.regexp_replace(F.col("true_total_raw"), r"[.,]", ""))
    .select("idx", "true_total_raw", "true_total")
)

labels.show(10, truncate=False)
print("receipts with a parsed total:", labels.filter("true_total is not null").count())

/home/jovyan/receipt-env/lib/python3.11/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


+---+--------------+----------+
|idx|true_total_raw|true_total|
+---+--------------+----------+
|0  |1,591,600     |1591600   |
|750|583,968       |583968    |
|427|726,495       |726495    |
|303|836,000       |836000    |
|544|196,350.00    |19635000  |
|787|246.073       |246073    |
|748|127,678       |127678    |
|203|237,997       |237997    |
|65 |650,100       |650100    |
|777|1,303,112     |1303112   |
+---+--------------+----------+
only showing top 10 rows


receipts with a parsed total: 779


In [7]:
# NOTEBOOK: 03_spark.ipynb  -- cell 6 (smarter total normalization + audit)
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import json, re

def parse_total(gt_string):
    # returns the total as an integer number of the smallest currency unit,
    # handling both thousands separators and a possible decimal .00
    try:
        raw = json.loads(gt_string)["gt_parse"]["total"]["total_price"]
    except Exception:
        return None
    if raw is None:
        return None
    s = raw.strip()
    # case: ends in .00 or .XX  -> treat final .dd as decimals, drop them, keep integer part
    m = re.match(r"^([\d.,]+?)\.(\d{2})$", s)
    if m:
        integer_part = re.sub(r"[.,]", "", m.group(1))
        return integer_part                    # drop the cents, keep whole units
    # otherwise: all , and . are thousands separators -> strip them
    return re.sub(r"[.,]", "", s)

parse_total_udf = F.udf(parse_total, StringType())

labels = (df.select("idx", "ground_truth")
    .withColumn("true_total_raw",
        F.udf(lambda g: (json.loads(g)["gt_parse"].get("total") or {}).get("total_price")
              if g else None, StringType())("ground_truth"))
    .withColumn("true_total", parse_total_udf("ground_truth"))
    .select("idx", "true_total_raw", "true_total"))

# audit: show the tricky ones (those that had a . in them)
print("---- receipts whose raw total contains a period ----")
labels.filter(F.col("true_total_raw").contains(".")).show(20, truncate=False)

print("receipts with a parsed total:", labels.filter("true_total is not null").count())

---- receipts whose raw total contains a period ----


+---+--------------+----------+
|idx|true_total_raw|true_total|
+---+--------------+----------+
|544|196,350.00    |196350    |
|787|246.073       |246073    |
|584|79,750.00     |79750     |
|293|466.620       |466620    |
|191|33.500        |33500     |
|538|287.595       |287595    |
|569|34.000        |34000     |
|254|259.298       |259298    |
|357|245.201       |245201    |
|162|Rp 41.580     |Rp 41580  |
|478|Rp. 230.999   |Rp 230999 |
|62 |3.600.000     |3600000   |
|361|282.000       |282000    |
|633|68.500        |68500     |
|48 |Rp 73.450     |Rp 73450  |
|261|134.000       |134000    |
|794|1.220.500     |1220500   |
|58 |35.000,00     |3500000   |
|328|228.000       |228000    |
|774|172.500       |172500    |
+---+--------------+----------+
only showing top 20 rows


26/08/10 18:20:28 ERROR Executor: Exception in task 4.0 in stage 27.0 (TID 1795)
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/tmp/ipykernel_1309/699598616.py", line 15, in parse_total
AttributeError: 'list' object has no attribute 'strip'
 SQLSTATE: 38000
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:665)
	at org.apache.spark.sql.execution.python.PythonArrowOutput$$anon$1.read(PythonArrowOutput.scala:142)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:602)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:611)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggr

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "/tmp/ipykernel_1309/699598616.py", line 15, in parse_total
AttributeError: 'list' object has no attribute 'strip'

In [8]:
# NOTEBOOK: 03_spark.ipynb  -- cell 6 (robust total normalization)
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import json, re

def raw_total(gt_string):
    """Pull total_price out of the label, coping with str, list, or missing."""
    try:
        t = json.loads(gt_string)["gt_parse"].get("total")
    except Exception:
        return None
    if not t:
        return None
    val = t.get("total_price")
    if isinstance(val, list):          # some receipts store it as a list
        val = val[0] if val else None
    return str(val) if val is not None else None

def norm_total(gt_string):
    """Normalize to a plain integer string of whole currency units."""
    s = raw_total(gt_string)
    if s is None:
        return None
    s = s.strip()
    # drop currency letters/symbols like 'Rp', 'Rp.', spaces -> keep digits , .
    s = re.sub(r"[^\d.,]", "", s)
    # if it ends in .dd or ,dd (two-digit cents), drop those cents
    m = re.match(r"^(.*)[.,](\d{2})$", s)
    core = m.group(1) if m else s
    # remaining , and . are thousands separators -> remove
    digits = re.sub(r"[.,]", "", core)
    return digits if digits.isdigit() else None

raw_udf  = F.udf(raw_total,  StringType())
norm_udf = F.udf(norm_total, StringType())

labels = (df.select("idx", "ground_truth")
    .withColumn("true_total_raw", raw_udf("ground_truth"))
    .withColumn("true_total",     norm_udf("ground_truth"))
    .select("idx", "true_total_raw", "true_total"))

print("---- rows whose raw total had a period or letters ----")
labels.filter(F.col("true_total_raw").rlike(r"[.,A-Za-z]")).show(25, truncate=False)
print("receipts with a parsed total:", labels.filter("true_total is not null").count())

---- rows whose raw total had a period or letters ----
+---+--------------+----------+
|idx|true_total_raw|true_total|
+---+--------------+----------+
|0  |1,591,600     |1591600   |
|750|583,968       |583968    |
|427|726,495       |726495    |
|303|836,000       |836000    |
|544|196,350.00    |196350    |
|787|246.073       |246073    |
|748|127,678       |127678    |
|203|237,997       |237997    |
|65 |650,100       |650100    |
|777|1,303,112     |1303112   |
|266|1,241,790     |1241790   |
|674|1,241,790     |1241790   |
|527|777,840       |777840    |
|321|1,802,900     |1802900   |
|124|1,092,542     |1092542   |
|475|1,096,040     |1096040   |
|567|100,100       |100100    |
|103|1,096,040     |1096040   |
|297|300,300       |300300    |
|711|1,092,542     |1092542   |
|277|2,307,021     |2307021   |
|149|53,020        |53020     |
|623|441,350       |441350    |
|701|3,703,791     |3703791   |
|505|54,950        |54950     |
+---+--------------+----------+
only showing top 

receipts with a parsed total: 779


In [9]:
# NOTEBOOK: 03_spark.ipynb  -- cell 7 (ceiling metric: is the true total present in the OCR?)
from pyspark.sql import functions as F

# reuse the cleaned word table from cell 4 -> `clean`
# make a normalized digit version of each OCR token to compare against labels
ocr_prices = (clean
    .withColumn("ocr_digits", F.regexp_replace("text", r"[^\d]", ""))
    .filter(F.col("ocr_digits") != "")
    .select("idx", "ocr_digits"))

# per receipt, collect the set of all numeric strings the OCR saw
ocr_by_receipt = (ocr_prices
    .groupBy("idx")
    .agg(F.collect_set("ocr_digits").alias("ocr_numbers")))

# join to labels and check if the true total is among them
check = (labels.filter("true_total is not null")
    .join(ocr_by_receipt, "idx", "left")
    .withColumn("total_found",
        F.array_contains(F.col("ocr_numbers"), F.col("true_total"))))

found = check.filter("total_found").count()
total = check.count()
print(f"true total present in OCR: {found} / {total}  ({100*found/total:.1f}%)")

# show a few misses to understand WHY the OCR missed them
print("---- examples where the total was NOT found ----")
check.filter("~total_found").select("idx", "true_total", "ocr_numbers").show(5, truncate=60)

{"ts": "2026-08-10 18:21:51.509", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"~total_found\" due to data type mismatch: The first parameter requires the \"INTEGRAL\" type, however \"total_found\" has the type \"BOOLEAN\". SQLSTATE: 42K09", "context": {"errorClass": "DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o279.filter.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"~total_found\" due to data type mismatch: The first parameter requires the \"INTEGRAL\" type, however \"total_found\" has the type \"BOOLEAN\". SQLSTATE: 42K09; line 1 pos 0;\n'Filter ~total_found#206\n+- Project [idx#8L, true_total_raw#170, true_total#172, ocr_numbers#198, array_contains(ocr_numbers#198, true_total#172) AS total_found#206]\n   +- Project [idx#8L, true_total_raw#170, true_total#172, ocr_numbers#198]

true total present in OCR: 498 / 779  (63.9%)
---- examples where the total was NOT found ----


AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "~total_found" due to data type mismatch: The first parameter requires the "INTEGRAL" type, however "total_found" has the type "BOOLEAN". SQLSTATE: 42K09; line 1 pos 0;
'Filter ~total_found#206
+- Project [idx#8L, true_total_raw#170, true_total#172, ocr_numbers#198, array_contains(ocr_numbers#198, true_total#172) AS total_found#206]
   +- Project [idx#8L, true_total_raw#170, true_total#172, ocr_numbers#198]
      +- Join LeftOuter, (idx#8L = idx#203L)
         :- Filter isnotnull(true_total#172)
         :  +- Project [idx#8L, true_total_raw#170, true_total#172]
         :     +- Project [idx#8L, ground_truth#7, true_total_raw#170, norm_total(ground_truth#7)#171 AS true_total#172]
         :        +- Project [idx#8L, ground_truth#7, raw_total(ground_truth#7)#169 AS true_total_raw#170]
         :           +- Project [idx#8L, ground_truth#7]
         :              +- Relation [ground_truth#7,idx#8L,ocr#9] json

In [10]:
# NOTEBOOK: 03_spark.ipynb  -- cell 7 (fixed: exact AND substring match)
from pyspark.sql import functions as F

ocr_prices = (clean
    .withColumn("ocr_digits", F.regexp_replace("text", r"[^\d]", ""))
    .filter(F.col("ocr_digits") != "")
    .select("idx", "ocr_digits"))

ocr_by_receipt = (ocr_prices
    .groupBy("idx")
    .agg(F.collect_set("ocr_digits").alias("ocr_numbers"),
         F.concat_ws(" ", F.collect_list("ocr_digits")).alias("ocr_blob")))

check = (labels.filter("true_total is not null")
    .join(ocr_by_receipt, "idx", "left")
    # exact match: the total is its own token
    .withColumn("total_exact", F.array_contains(F.col("ocr_numbers"), F.col("true_total")))
    # looser: the total digits appear inside the concatenated OCR numbers
    .withColumn("total_substr", F.col("ocr_blob").contains(F.col("true_total"))))

n = check.count()
exact = check.filter(F.col("total_exact")).count()
substr = check.filter(F.col("total_substr")).count()
print(f"receipts checked: {n}")
print(f"exact-token match:   {exact} ({100*exact/n:.1f}%)")
print(f"substring match:     {substr} ({100*substr/n:.1f}%)")

print("---- misses (total not found even loosely) ----")
check.filter(~F.col("total_substr")).select("idx", "true_total", "ocr_numbers").show(5, truncate=60)

receipts checked: 779
exact-token match:   498 (63.9%)
substring match:     523 (67.1%)
---- misses (total not found even loosely) ----


+---+----------+------------------------------------------------------------+
|idx|true_total|                                                 ocr_numbers|
+---+----------+------------------------------------------------------------+
|  0|   1591600|[6500, 4000, 18, 100950, 850, 44, 2, 45, 591600, 180, 36,...|
|  4|     48000|             [000, 4364, 1000, 3, 4, 43636, 143635, 4800, 0]|
|  6|     61799|[1000, 56181, 61, 3, 626, 130, 434, 5618, 13000, 1, 4, 43...|
| 10|     25000|                                    [250, 25, 5000, 3000, 2]|
| 11|    250107|                     [250, 84, 27, 3, 707, 4, 334, 30, 7, 8]|
+---+----------+------------------------------------------------------------+
only showing top 5 rows


In [11]:
# NOTEBOOK: 03_spark.ipynb  -- cell 8 (confirm fragmentation: see raw tokens for a miss)
from pyspark.sql import functions as F

# look at receipt 10 (true total 25000) raw OCR tokens, in reading order
(clean.filter(F.col("idx") == 10)
      .select("text", "conf")
      .show(60, truncate=False))

+----------------+-------------------+
|text            |conf               |
+----------------+-------------------+
|"iet Hilk Caffee|0.48455845161895833|
|2,Qoo           |0.13494048252997143|
|'hot            |0.33128392696380615|
|Sntatal         |0.14093603716964978|
|25, CCC         |0.5494994848797731 |
|25.dw0          |0.39581842901417824|
|CUSH            |0.47227569871458946|
|3,000           |0.3657774987705879 |
|Kenbalian       |0.6005351715955943 |
|5,000           |0.5028340679324176 |
+----------------+-------------------+



In [12]:
# NOTEBOOK: 03_spark.ipynb  -- cell 9 (honest ceiling test: re-OCR failures at higher res)
import numpy as np, easyocr
from datasets import load_dataset
from PIL import Image

ds = load_dataset("naver-clova-ix/cord-v2")["train"]
reader = easyocr.Reader(["en"], gpu=True)

# receipts we know failed the substring test earlier
fail_ids = [0, 4, 6, 10, 11]

def read_digits(img, scale=1.0):
    if scale != 1.0:
        w, h = img.size
        img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
    raw = reader.readtext(np.array(img), detail=1)
    nums = []
    for _, text, conf in raw:
        d = "".join(ch for ch in text if ch.isdigit())
        if d:
            nums.append((d, round(conf, 2)))
    return nums

for i in fail_ids:
    img = ds[i]["image"]
    base = read_digits(img, 1.0)
    up   = read_digits(img, 2.0)   # 2x upscaled
    print(f"\n--- receipt {i} ---")
    print("1x digits:", [d for d,_ in base])
    print("2x digits:", [d for d,_ in up])
    

/home/jovyan/receipt-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- receipt 0 ---
1x digits: ['75000', '125000', '3', '37', '24', '70000', '3', '3', '6500', '180', '2900', '850', '2', '360', '2', '36', '4000', '1', '70', '3', '366', '92', '2', '44', '1', '320', '4000', '1', '1', '440', '18', '1346000', '100950', '144695', '45', '1', '591600']
2x digits: ['75000', '125000', '37000', '1', '24', '70000', '83', '65', '1', '1800', '2900', '3', '85', '36', '40', '1', '376', '000', '920', '44', '32', '1', '40', '44', '1800', '1', '3462', '100950', '144695', '45', '1', '591', '600', '36']

--- receipt 4 ---
1x digits: ['3', '3', '4', '143635', '43636', '3', '43636', '1000', '4364', '4800', '000', '0']
2x digits: ['2', '3', '3', '143635', '43636', '3', '43636', '1000', '4364', '3', '40', '00', '0', '3']

--- receipt 6 ---
1x digits: ['7', '1', '181', '43184', '1', '13000', '130', '56181', '5618', '1000', '61', '4', '626', '1', '434', '3']
2x digits: ['44', '1', '43181', '43184', '1', '13000', '13', '56181', '5618', '1000', '617', '4', '6200', '201', '1', '

In [13]:
# NOTEBOOK: 03_spark.ipynb  -- cell 10 (label each word: is it the total?)
from pyspark.sql import functions as F

# each OCR word, with its digits-only version
words_lab = (clean
    .withColumn("ocr_digits", F.regexp_replace("text", r"[^\d]", ""))
    .select("idx", "text", "conf", "box", "ocr_digits"))

# bring in the true total per receipt
words_lab = words_lab.join(
    labels.select("idx", "true_total"), "idx", "left")

# label = "TOTAL" if this word's digits equal the receipt's true total, else "OTHER"
words_lab = words_lab.withColumn(
    "label",
    F.when((F.col("ocr_digits") != "") &
           (F.col("ocr_digits") == F.col("true_total")), "TOTAL")
     .otherwise("OTHER"))

# quick sanity check: how many TOTAL vs OTHER labels?
words_lab.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|TOTAL|  912|
|OTHER|13775|
+-----+-----+



In [14]:
# NOTEBOOK: 03_spark.ipynb  -- cell 11 (save labeled data for the model)
# keep only what the model needs, write to processed as parquet
out = words_lab.select("idx", "text", "conf", "box", "label")

out.write.mode("overwrite").parquet("data/processed/labeled_words.parquet")

# confirm it wrote and can be read back
check = spark.read.parquet("data/processed/labeled_words.parquet")
print("rows written:", check.count())
check.show(5, truncate=False)

rows written: 14687
+---+----------+-------------------+----------------------------------------------------------------+-----+
|idx|text      |conf               |box                                                             |label|
+---+----------+-------------------+----------------------------------------------------------------+-----+
|0  |Nasi      |0.9998831152915955 |[[300.0, 366.0], [354.0, 366.0], [354.0, 392.0], [300.0, 392.0]]|OTHER|
|0  |Campur    |0.992746930021226  |[[362.0, 366.0], [440.0, 366.0], [440.0, 390.0], [362.0, 390.0]]|OTHER|
|0  |Bali      |0.9818131327629089 |[[446.0, 364.0], [498.0, 364.0], [498.0, 388.0], [446.0, 388.0]]|OTHER|
|0  |75,000    |0.41665860322926634|[[542.0, 360.0], [618.0, 360.0], [618.0, 386.0], [542.0, 386.0]]|OTHER|
|0  |Bbk Bengil|0.8182316162725418 |[[304.0, 392.0], [428.0, 392.0], [428.0, 420.0], [304.0, 420.0]]|OTHER|
+---+----------+-------------------+----------------------------------------------------------------+-----+
only sho

In [15]:
# NOTEBOOK: 03_spark.ipynb  -- check if earlier variables still exist
try:
    print("clean rows:", clean.count())
    print("labels rows:", labels.count())
    print("ready: yes")
except NameError as e:
    print("variables gone, need to re-run earlier cells:", e)

clean rows: 14687
labels rows: 800
ready: yes


In [16]:
# NOTEBOOK: 03_spark.ipynb  -- cell 12 (analysis across all receipts)
from pyspark.sql import functions as F

print("="*50)
print("1. DATASET SIZE")
print("="*50)
print("receipts:", clean.select("idx").distinct().count())
print("total OCR word-tokens:", clean.count())

print("\n" + "="*50)
print("2. OCR CONFIDENCE DISTRIBUTION")
print("="*50)
clean.select(
    F.round(F.avg("conf"), 3).alias("mean_conf"),
    F.round(F.expr("percentile_approx(conf, 0.5)"), 3).alias("median_conf"),
    F.round(F.min("conf"), 3).alias("min_conf"),
    F.round(F.max("conf"), 3).alias("max_conf"),
).show()

# how much of the OCR is low-confidence?
low = clean.filter(F.col("conf") < 0.5).count()
tot = clean.count()
print(f"low-confidence tokens (<0.5): {low} / {tot} ({100*low/tot:.1f}%)")

print("="*50)
print("3. WORDS PER RECEIPT")
print("="*50)
per = clean.groupBy("idx").count()
per.select(
    F.round(F.avg("count"), 1).alias("avg_words"),
    F.min("count").alias("min_words"),
    F.max("count").alias("max_words"),
).show()

print("="*50)
print("4. TOTAL VALUE STATISTICS (from labels)")
print("="*50)
labels.filter("true_total is not null").select(
    F.round(F.avg(F.col("true_total").cast("double")), 0).alias("avg_total"),
    F.min(F.col("true_total").cast("double")).alias("min_total"),
    F.max(F.col("true_total").cast("double")).alias("max_total"),
).show()

1. DATASET SIZE


receipts: 800


total OCR word-tokens: 14687

2. OCR CONFIDENCE DISTRIBUTION


+---------+-----------+--------+--------+
|mean_conf|median_conf|min_conf|max_conf|
+---------+-----------+--------+--------+
|    0.654|       0.73|     0.0|     1.0|
+---------+-----------+--------+--------+



low-confidence tokens (<0.5): 4814 / 14687 (32.8%)
3. WORDS PER RECEIPT


+---------+---------+---------+
|avg_words|min_words|max_words|
+---------+---------+---------+
|     18.4|        5|       86|
+---------+---------+---------+

4. TOTAL VALUE STATISTICS (from labels)


+---------+---------+---------+
|avg_total|min_total|max_total|
+---------+---------+---------+
| 132502.0|   3100.0|3850000.0|
+---------+---------+---------+

